**Elèves**: Antoine Rustenholz, Jasmin Neveu, Aziz Seghaier

# Marathon de Paris 2026

L'objectif de ce projet est d'analyser les données du jeu de Paris 2026.


## Collecte des données

Les données ont été collectées via deux appels successifs à une API.

Un premier appel nous a permis de récupérer l'ensemble des caractéristiques 
des athlètes (dossard, nom, prénom, sexe, catégorie, nationalité, temps final et 
classements). Ces données ne contenaient pas les temps intermédiaires.

Nous avons ensuite parcouru l'ensemble des identifiants athlètes pour effectuer 
un appel individuel par coureur, récupérant ainsi ses temps de passage à chacun 
des 10 points de contrôle (splits).

Les deux tables ont finalement été jointes sur l'identifiant athlète pour constituer 
le jeu de données final, soit environ 57 000 coureurs avec l'ensemble de leurs 
caractéristiques et splits.

## Structure du notebook

- **Statistiques descriptives**
- **Modélisation** : prédiction du temps final à partir des caractéristiques 
  de l'athlète et de ses vitesses sur les premiers splits


# 1. Statistiques descriptives

In [ ]:
import stat_desc as sd
import polars as pl

df = pl.read_parquet("data/donnees_finales.parquet")

## 1.1 Les performances générales des coureurs

In [ ]:
total = df.shape[0]
prop_H = round(df.filter(pl.col("sex") == "M").shape[0] / df.shape[0] * 100, 2)
print(f"Nombre total de finishers : {total}")
print(f"Proportion d'hommes : {prop_H}%")

def calcul_temps(stat: str, sex: str = None) -> str:
    """Calcule le temps moyen ou médian global ou par sexe"""

    df_filtered = df.filter(pl.col("sex") == sex) if sex is not None else df

    if stat == "mean":
        valeur = df_filtered["realTime"].mean()
    else:
        valeur = df_filtered["realTime"].median()

    valeur = int(valeur)
    h = valeur // 3600
    m = (valeur % 3600) // 60
    s = valeur % 60
    return f"{h}:{m:02d}:{s:02d}"

print(f"Temps moyen global : {calcul_temps("mean")}")
print(f"Temps médian global : {calcul_temps("median")}")
print(f"Temps moyen homme : {calcul_temps("mean", "M")}")
print(f"Temps moyen femme : {calcul_temps("mean", "F")}")

Pour cette édition 2026, 57477 coureurs ont fini le parcours du marathon de Paris. Parmi eux, plus de deux tiers sont des hommes. On remarque un écart d'environ 30 minutes entre les temps moyens masculin et féminin. Cela se vérifie sur la distribution des temps par sexe ci-dessous :

In [ ]:
sd.distrib_tps(0.99)

Sur ce graphique, on observe des pics réguliers proches de valeurs précises comme 3h, 3h30, 4h ou encore 4h40 pour les plus nets. Ces pics peuvent être vus comme un effet de seuil : de nombreux coureurs ont un certain objectif de temps et font en sorte d'arriver juste avant leur objectif.
Concernant ces seuils justement, voici la répartition des hommes et des femmes selon des seuils de 30 mniutes :

In [ ]:
sd.repart_seuils()

D'après ce tableau, on retrouve bien la même tendance chez les hommes et les femmes mais décalée d'environ 30 minutes de moins pour les femmes.

## 1.2 L'âge, un facteur si important ?

Tout type de profil peut s'attaquer à un marathon. En particulier, on peut retrouver tous les âges. Voici une courbe représentant l'évolution du temps moyen par sexe selon l'âge des athlètes :

In [ ]:
sd.evol_tps_age()

On remarque que, en moyenne, moins de 30 minutes séparent la catégorie senior (23-34 ans) et la catégorie M4 (55-59 ans). A part pour les valeurs extrêmes, c'est-à-dire à partir de 70 ans, les temps moyes diminuent relativement faiblement avec l'âge.

## 1.3 Les performances des internationaux

In [ ]:
nb_pays = df.select(pl.col("nationality")).n_unique()
print(f"Nombre de nationalités représentées : {nb_pays}")

nb_francais = df.filter(pl.col("nationality") == "FR").n_unique()
prop_francais = round(nb_francais / total * 100, 2)
print(f"Nombre de français participants : {nb_francais} soit {prop_francais}% du peloton")

Comment tout grand marathon, celui de Paris attire de nombreux étrangers. Malgré les 143 nations représentées, les français restent très largement majoritaires en représentant 71.9% de tous les concurrents. Pour ce qui est des autres nations représentées, en voici les 10 plus nombreuses :

In [ ]:
sd.repart_pays()

Il peut être intéressant d'étudier si les nations les plus représentées sont aussi celles dont proviennent les meilleurs athlètes. Pour cela, voici un tableau représentant les nations les plus représentées parmi le premier centile, excluant la France qui, à nouveau, est sur-représentée. On remarque alors que ces nations sont très peu représentées.

In [ ]:
sd.meilleurs_pays()

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

df_ro = pl.read_parquet(r"data\donnees_finales.parquet")

correlation = df_ro.select(
    pl.corr("realTime", "retardTime")
).item()

# 2. Création du Scatter Plot
plt.figure(figsize=(10, 6))
plt.scatter(df_ro["retardTime"], df_ro["realTime"], alpha=0.5, color="blue", s=10)

plt.title(f"Scatter Plot : realTime vs retard (Corrélation: {correlation:.4f})")
plt.xlabel("retard")
plt.ylabel("temps réel")
plt.grid(True, linestyle='--', alpha=0.7)

# 2. Modélisation
L'objectif est de prédire le temps d'arrivée officiel (`realTime`) à partir des caractéristiques des athlètes et des données des premiers splits intermédiaires.

**Plan**
1. Data Engineering
2. Entraînement — Ridge et ElasticNet
3. Évaluation sur le jeu de test
4. Intervalles de confiance des coefficients (Bootstrap)
5. Visualisations

## 0. Imports et configuration

## 1. Data Engineering

Le pipeline effectue les opérations suivantes dans l'ordre :
- **Suppression des DNF** : tout athlète avec un split manquant est retiré
- **Suppression des fuites** : rankings globaux, pace/speed globaux (calculés a posteriori)
- **Regroupement des nationalités rares** : les nationalités représentées par moins de `NATIONALITY_MIN_COUNT` athlètes sont regroupées dans la catégorie `OTHER`
- **One-Hot Encoding** : `sex`, `category`, `nationality`

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import polars as pl

from src.model.config import (
    CONFIDENCE_LEVEL,
    N_BOOTSTRAP,
    N_SPLITS,
    NATIONALITY_MIN_COUNT,
)
from src.model.data_engineering import build_dataset, split_features_target
from src.model.train import evaluate_model, prepare_arrays, train_elasticnet, train_ridge
from src.model.bootstrap import bootstrap_confidence_intervals, significant_features
from src.model.visualization import (
    plot_confidence_intervals,
    plot_metrics_comparison,
    plot_predicted_vs_actual,
    plot_residuals,
)

print(f"Paramètres actifs :")
print(f"  Splits utilisés     : {N_SPLITS}")
print(f"  Seuil nationalités  : {NATIONALITY_MIN_COUNT}")
print(f"  Itérations bootstrap: {N_BOOTSTRAP}")
print(f"  Niveau de confiance : {CONFIDENCE_LEVEL:.0%}")

In [ ]:
df = build_dataset(n_splits=N_SPLITS, nationality_min_count=NATIONALITY_MIN_COUNT)
print(f"Dimensions du dataset final : {df.shape}")
print(f"\nAperçu des colonnes :")
print(df.columns)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['realTime'].to_numpy(), bins=80, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Temps réel (secondes)')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution du temps réel (variable cible)')
plt.tight_layout()
plt.show()

In [ ]:
X, y = split_features_target(df)
feature_names = X.columns
X_train, X_test, y_train, y_test = prepare_arrays(X, y)

print(f"Taille train : {X_train.shape[0]} | Taille test : {X_test.shape[0]}")
print(f"Nombre de features : {X_train.shape[1]}")

## 2. Entraînement

Les deux modèles utilisent un pipeline `StandardScaler → Régression` pour que la régularisation s'applique équitablement sur toutes les features. Les hyperparamètres sont sélectionnés par validation croisée à 5 folds (critère : RMSE).

In [ ]:
print("Entraînement Ridge...")
ridge_grid = train_ridge(X_train, y_train)
print(f"  Meilleur alpha : {ridge_grid.best_params_['model__alpha']}")

In [ ]:
print("Entraînement ElasticNet...")
enet_grid = train_elasticnet(X_train, y_train)
print(f"  Meilleur alpha    : {enet_grid.best_params_['model__alpha']}")
print(f"  Meilleur l1_ratio : {enet_grid.best_params_['model__l1_ratio']}")

## 3. Évaluation sur le jeu de test

In [ ]:
ridge_metrics = evaluate_model(ridge_grid, X_test, y_test)
print("Ridge")
print(f"  RMSE : {ridge_metrics['rmse']:.1f} secondes")
print(f"  MAE  : {ridge_metrics['mae']:.1f} secondes")
print(f"  R²   : {ridge_metrics['r2']:.4f}")


In [ ]:
ridge_metrics = evaluate_model(ridge_grid, X_test, y_test)
enet_metrics  = evaluate_model(enet_grid,  X_test, y_test)

print("Ridge")
print(f"  RMSE : {ridge_metrics['rmse']:.1f} secondes")
print(f"  MAE  : {ridge_metrics['mae']:.1f} secondes")
print(f"  R²   : {ridge_metrics['r2']:.4f}")

print("\nElasticNet")
print(f"  RMSE : {enet_metrics['rmse']:.1f} secondes")
print(f"  MAE  : {enet_metrics['mae']:.1f} secondes")
print(f"  R²   : {enet_metrics['r2']:.4f}")

In [ ]:
plot_metrics_comparison({
    'Ridge': ridge_metrics,
    'ElasticNet': enet_metrics,
})
plt.show()

In [ ]:
y_pred_ridge = ridge_grid.predict(X_test)
plot_predicted_vs_actual(y_test, y_pred_ridge, model_name='Ridge')
plt.show()

In [ ]:
plot_residuals(y_test, y_pred_ridge, model_name='Ridge')
plt.show()

## 4. Intervalles de confiance des coefficients (Bootstrap)

**Pourquoi le bootstrap et pas les IC analytiques ?**  
La régularisation Ridge biaise intentionnellement les coefficients — les formules MCO classiques basées sur $(X^\top X)^{-1}$ ne s'appliquent plus. ElasticNet n'a pas d'IC analytiques standards. Le bootstrap est non-paramétrique et valide dans les deux cas : on rééchantillonne l'ensemble d'entraînement `N_BOOTSTRAP` fois, on réentraîne le pipeline complet à chaque itération, et on construit l'IC à partir des percentiles empiriques de la distribution des coefficients.

In [ ]:
print(f"Bootstrap Ridge ({N_BOOTSTRAP} itérations)...")
ridge_ci = bootstrap_confidence_intervals(
    pipeline=ridge_grid.best_estimator_,
    X_train=X_train,
    y_train=y_train,
    feature_names=list(feature_names),
)
print("Terminé.")

In [ ]:
print(f"Bootstrap ElasticNet ({N_BOOTSTRAP} itérations)...")
enet_ci = bootstrap_confidence_intervals(
    pipeline=enet_grid.best_estimator_,
    X_train=X_train,
    y_train=y_train,
    feature_names=list(feature_names),
)
print("Terminé.")

In [ ]:
# Features significativement différentes de 0 pour Ridge
sig_ridge = significant_features(ridge_ci)
print(sig_ridge)
print(f"Features significatives (Ridge) : {len(sig_ridge)} / {len(ridge_ci)}")

## 5. Visualisation des intervalles de confiance

In [ ]:
plot_confidence_intervals(
    ridge_ci,
    top_n=30,
    model_name='Ridge',
    confidence_level=CONFIDENCE_LEVEL,
)
plt.show()

In [ ]:
plot_confidence_intervals(
    enet_ci,
    top_n=30,
    model_name='ElasticNet',
    confidence_level=CONFIDENCE_LEVEL,
)
plt.show()

## 6. Changer le nombre de splits

Pour reproduire l'analyse avec un nombre différent de splits, il suffit de modifier `N_SPLITS` dans `config.py` et de relancer ce notebook. Exemple ci-dessous avec 5 splits.

In [ ]:
# Exemple : relancer le pipeline avec 5 splits
N_SPLITS_DEMO = 5

df_5 = build_dataset(n_splits=N_SPLITS_DEMO)
X_5, y_5 = split_features_target(df_5)
X_train_5, X_test_5, y_train_5, y_test_5 = prepare_arrays(X_5, y_5)

ridge_grid_5 = train_ridge(X_train_5, y_train_5)
metrics_5 = evaluate_model(ridge_grid_5, X_test_5, y_test_5)

print(f"Ridge avec {N_SPLITS_DEMO} splits")
print(f"  RMSE : {metrics_5['rmse']:.1f} s")
print(f"  R²   : {metrics_5['r2']:.4f}")